# Blockchain di bitcoin

## Modello Permissionless
Come accennato precedentemente l'architettura di bitcoin si basa su un sistema distribuito permissionless.

Ogni nodo del sistema presenta un vettore di blocchi, dove ogni blocco è composto da:  
- **Block header**: 
  - Contenente le informazioni che identificano un blocco
- **Block data**: 
  - L'insieme delle transazioni  

Detto questo il protocollo sotto Bitcoin fa in modo che valgano:
1. **Eventual consistency**:
   - Tutti i nodi del sistema presentano lo stesso vettore dei blocchi, eccetto qualcuna delle ultime entry
2. **Liveness**: 
   - Quando un nodo onesto riceve una transazione *Tx*, prima o poi questa viene messa all'interno di un blocco
>Oss: La Liveness é la versione di Bitcoin della Validity

#### Sybil Attacks 

All'interno di un sistemma PermissionLess nulla vieta ad un agente corrotto di simulare un numero arbitrario di nodi da immettere all'interno della rete.

Per questo é necessario che il protocollo garantisca che: 
   - Un nodo onesto che entri all'interno della rete deve essere in grado di determinare quale sia il *vettore corretto*

---

## Funzione Hash Crittogafica

Una funzione **Hash Crittografica** è una funzione:

$$
\mathcal{H}: \{0,1\}^{*} \rightarrow \{0,1\}^{\ell}
$$

Dove valgono le seguenti proprietà:

1. **Preimage Resistance**:
   - Dato $y \in \{0,1\}^{\ell}$ è difficile trovare $x \in \{0,1\}^{*}$ tale che $\mathcal{H}(x) = y$

2. **Second Preimage Resistance**:
   - Dato $x_1 \in \{0,1\}^{*}$ è difficile trovare $x_2 \in \{0,1\}^{*}$ tale che $\mathcal{H}(x_1) = \mathcal{H}(x_2)$

3. **Collision Resistance**:
   - È difficile trovare $x_1, x_2 \in \{0,1\}^{*}$ tali che $x_1 \neq x_2$ e $\mathcal{H}(x_1) = \mathcal{H}(x_2)$

>Oss: In bitcoin si utilizza la funzione $ Sha256:\{0,1\}^{*} \rightarrow \{0,1\}^{256}$

>Nota: Con difficile si intende l'impossibilità di risoluzione senza l'utilizzo di brute force 

---

## Header del blocco

Prima di introdurre la proof of work bisogna parlare della struttura dell'*header* di un blocco.

L'header di ogni blocco da **80 byte** divisi fra:
- Version:
  - il numero di versione, scritto in 4 byte
- Prev-hash:
  - l'Hash del blocco precedente, scritto in 32 byte
- Data-hash(O Merkle-Root):
  - l'Hash del contenuto del blocco, scritto in 32 byte
- Timestamp:
  - data e ora in cui viene creato il blocco rappresentata in unix time, scritto in 4 byte 
- Target:
  - un'intero fondamentale per far funzionare la proof of work, scritto in 4 byte
- Nonce:
  - un'intero anch'esso fondamentale per la proof of work, scritto in 4 byte

>Oss: Da notare che il nonce è l'unico valore veramente arbitrario


In [ ]:
from io import BytesIO 
from hashlib import sha256
from datetime import datetime

class Block:

    def __init__(self, version, prev_hash, merkle_root, timestamp, bits, nonce):
        self.version = version #Versione del blocco
        self.prev_hash = prev_hash #Hash del blocco precedente
        self.merkle_root = merkle_root #Contenuto del blocco
        self.timestamp = timestamp #Instante di tempo in cui viene effettuato l'hash del blocco
        self.bits = bits #Target del blocco
        self.nonce = nonce #4 Byte aggiunti per trovare l'hash del blocco  

    @classmethod
    def parse(cls, byte_string): #Funzione che legge l'header di un blocco e lo spezza nelle sue componentistiche
        reader = BytesIO(byte_string)
        version = reader.read(4)
        prev_hash = reader.read(32)
        merkle_root = reader.read(32)
        timestamp = reader.read(4)
        bits = reader.read(4)
        nonce = reader.read(4)
        return cls(version[::-1], prev_hash[::-1], merkle_root[::-1], timestamp[::-1], bits[::-1], nonce[::-1])



---

## Proof of Work 

Il protocollo di Bitcoin utilizza la **Proof of Work** per far valere l'*Eventual consistency* e la *Liveness*. 

La logica dietro la *Proof of Work* é abbastanza semplice:
- Siano:
  - $t \le 2^{256}$ il target su cui tutti i nodi si mettono d'accordo  
  - $blk$ il nuovo blocco da inserire in coda alla blockchain, con header $blk.header$  
  - $h = hash(blk.header)$ l'hash del blocco  


$$
\text{Un blocco è valido} \\
\Updownarrow \\
h \le t \;\land\; blk.prev\_hash = hash(\text{blocco precedente})
$$



Da questa relazione si deduce che per generare un nuovo blocco valido è necessario trovare un input (tipicamente variando il **nonce**) tale che l’hash dell’header del blocco sia inferiore al target.

In pratica, i miner provano ripetutamente diversi valori di nonce (e altri campi variabili) e calcolano l’hash finché non trovano un valore valido.

---

Ad esempio, possiamo cercare un nonce tale che l’hash della stringa `"Francesco"` concatenata al nonce produca un valore con diversi zeri iniziali (in rappresentazione esadecimale o binaria).

In [ ]:
from hashlib import sha256

nonce = "16114492071"
text = 'Francesco ' + nonce

print(sha256(text.encode('utf-8')).hexdigest())

find = False
i = 1
while(not find):
    for tempNonce in range(2**(i-1),2**i):
        text = 'Francesco ' + str(tempNonce)
        print(tempNonce)
        hash = sha256(text.encode('utf-8')).hexdigest() 
        if hash[0:9] == "000000000":
            find = True
            print(hash)
    i += 1

Qui facciamo un'idea sul come vengono generati i nonce da aggiungere un blocco

In questo caso stiamo cercando un nonce per fare in modo che l'hash della della stringa 'Francesco' concatenata al nonce sia un interno con i primi nove valori uguali a 0

---

### Criterio di scelta del target

Il target viene scelto in modo tale che, in media, venga aggiunto un nuovo blocco ogni circa **10 minuti**.

Il ricalcolo del target avviene ogni **2016 blocchi** ($\approx 14$ giorni), per adattarsi alla potenza di calcolo della rete.

Più nello specifico:
- se i 2016 blocchi sono stati generati in meno di 14 giorni → la rete è troppo veloce → **il target viene abbassato (aumenta la difficoltà)**  
- se sono stati generati in più di 14 giorni → la rete è lenta → **il target viene alzato (diminuisce la difficoltà)**  

---

### Biforcazioni della catena

Può capitare che due blocchi validi e differenti tra loro, $A$ e $B$, vengano aggiunti quasi contemporaneamente alla blockchain.

In questo caso si creano due catene identiche tranne che per l’ultimo blocco.

Quando un nodo riceve entrambe le versioni, mantiene temporaneamente entrambe le catene, ma continuerà a estendere quella che accumula più lavoro computazionale.

Alla fine, la rete converge sulla **catena con la maggiore Proof of Work cumulativa** (spesso approssimata come la catena più lunga), mentre l’altra viene abbandonata.

---

All'interno del prossimo blocco viene mostrata una possibile implementazione di un blocco

In [ ]:
from io import BytesIO 
from hashlib import sha256

class Block:
    def __init__(self, version, prev_hash, merkle_root, timestamp, bits, nonce):
        self.version = version 
        self.prev_hash = prev_hash 
        self.merkle_root = merkle_root 
        self.timestamp = timestamp 
        self.bits = bits 
        self.nonce = nonce 
    
    @classmethod
    def parse(cls, byte_string): 
        reader = BytesIO(byte_string)
        version = reader.read(4)
        prev_hash = reader.read(32)
        merkle_root = reader.read(32)
        timestamp = reader.read(4)
        bits = reader.read(4)
        nonce = reader.read(4)
        return cls(version[::-1], prev_hash[::-1], merkle_root[::-1], timestamp[::-1], bits[::-1], nonce[::-1])

    def serialize(self): 
        return self.version[::-1] + self.prev_hash[::-1] + self.merkle_root[::-1] + self.timestamp[::-1] + self.bits[::-1] + self.nonce[::-1]

    def hash(self): 
        return sha256(sha256(self.serialize()).digest()).digest()[::-1]
    
    def target(self): 
        return int.from_bytes(self.bits[1:], "big") * 256**(self.bits[0] - 3)
    
    def is_valid_target(self): 
        return int.from_bytes(self.hash(), "big") <= self.target()

    def update_nonce(self, i):
        self.nonce = i.to_bytes(4, 'big')

    def now(self):
        return int(datetime.now().timestamp()).to_bytes(4, 'big')

    def __str__(self):
        out = dict()
        out['version'] = self.version.hex()
        out['prev_hash'] = self.prev_hash.hex()
        out['merkle_root'] = self.merkle_root.hex()
        out['timestamp'] = self.timestamp.hex()
        out['bits'] = self.bits.hex()
        out['nonce'] = int(self.nonce.hex(), 16)
        return out.__str__()
    

# 3. Codice di test portato FUORI dalla classe (nessuna indentazione)
header = "00e0a323fd73cb833ac1b091f339b14cb320f98fa7756d31fe520100000000\
          000000000093918fc1931493becee43e9ec9f4d12fbcb4e8815653163ea58e\
          cffe8c4e6a361f9edc6984060217395b599f"
blk = Block.parse(bytes.fromhex(header))

print("Block info:", blk)

# Usiamo .hex() altrimenti stampa i byte grezzi illeggibili
print("Hash:", blk.hash().hex()) 

print("Target:", hex(blk.target()))

print("Is valid?", blk.is_valid_target())

Ora facciamo vedere invece come viene effettivammente eseguita la **proof of work** andando a creare una blockchain fasulla

In [ ]:
GENESIS = "01000000000000000000000000000000000000000000000000000000\
           00000000000000003ba3edfd7a7b12b27ac72c3e67768f617fc81bc3\
           888a51323a9fb8aa4b1e5e4a29ab5f49ffff001d1dac2b7c"

blk = Block.parse(bytes.fromhex(GENESIS))

print(blk)
print(blk.hash().hex())
print()

height = 0
while True:
    blk.prev_hash = blk.hash()
    blk.bits = bytes.fromhex('1e02ffff')
    blk.timestamp = blk.now()
    height += 1
    tx = "PCD 25/26 block " + str(height) 
    blk.merkle_root = sha256(sha256(tx.encode('utf-8')).digest()).digest()
    i = 0
    blk.update_nonce(i)
    flag = blk.is_valid_target()
    while not flag:
        i += 1
        if i == 2**32:
            i = 0
            blk.timestamp = blk.now()
        blk.update_nonce(i)
        flag = blk.is_valid_target()

    print(blk)
    print(blk.hash().hex())
    print()